# Lab | Agent & Vector store

**Change the state union dataset and replicate this lab by updating the prompts accordingly.**

One such dataset is the [sonnets.txt](https://github.com/martin-gorner/tensorflow-rnn-shakespeare/blob/master/shakespeare/sonnets.txt) dataset or any other data of your choice from the same git.

# Combine agents and vector stores

This notebook covers how to combine agents and vector stores. The use case for this is that you've ingested your data into a vector store and want to interact with it in an agentic manner.

The recommended method for doing so is to create a `RetrievalQA` and then use that as a tool in the overall agent. Let's take a look at doing this below. You can do this with multiple different vector DBs, and use the agent as a way to route between them. There are two different ways of doing this - you can either let the agent use the vector stores as normal tools, or you can set `return_direct=True` to really just use the agent as a router.

## Create the vector store

In [28]:
#!pip install chromadb langchain langchain_community langchain_openai

In [29]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [30]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [31]:
# If you're using colab, run this
#os.environ['OPENAI_API_KEY'] = "YOUR_OPENAI_API_KEY"

In [32]:
llm = OpenAI(temperature=0)

In [33]:
from pathlib import Path

relevant_parts = []
for p in Path(".").absolute().parts:
    relevant_parts.append(p)
    if relevant_parts[-3:] == ["langchain", "docs", "modules"]:
        break
doc_path = str(Path(*relevant_parts) / "state_of_the_union.txt")

In [34]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings()
docsearch = Chroma.from_documents(texts, embeddings, collection_name="ruff-docs")

Created a chunk of size 2122, which is longer than the specified 1000
Created a chunk of size 3187, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 1256, which is longer than the specified 1000
Created a chunk of size 2321, which is longer than the specified 1000


In [35]:
state_of_union = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [36]:
from langchain_community.document_loaders import WebBaseLoader

In [37]:
loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")

In [38]:
pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [39]:
docs = loader.load()
ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

Created a chunk of size 2122, which is longer than the specified 1000
Created a chunk of size 3187, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 1256, which is longer than the specified 1000
Created a chunk of size 2321, which is longer than the specified 1000


## Create the Agent

In [40]:
# Import things that are needed generically
from langchain.agents import AgentType, Tool, initialize_agent
from langchain_openai import OpenAI

In [41]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

In [42]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [43]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)



> Entering new AgentExecutor chain...
 I should use the State of Union QA System to answer this question.
Action: State of Union QA System
Action Input: "What did Biden say about Ketanji Brown Jackson in the state of the union address?"
Observation:  I don't know, I'm sorry.
Thought: I should try rephrasing the question.
Action: State of Union QA System
Action Input: "What was Biden's mention of Ketanji Brown Jackson in the state of the union address?"
Observation:  I don't know.
Thought: I should try using more specific keywords.
Action: State of Union QA System
Action Input: "What was Biden's praise of Ketanji Brown Jackson in the state of the union address?"
Observation:  I don't know.
Thought: I should try using a different source.
Action: Ruff QA System
Action Input: "What did Biden say about Ketanji Brown Jackson in the state of the union address?"
Observation:  I don't know.
Thought: I should try using a different source.
Action: Ruff QA System
Action Input: "What was Biden's 

{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': "I'm sorry, I don't have enough information to answer this question."}

In [44]:
agent.invoke("Why use ruff over flake8?")



> Entering new AgentExecutor chain...
 Ruff is a python linter that has some unique features compared to flake8, so it may be useful in certain situations.
Action: Ruff QA System
Action Input: "Why use ruff over flake8?"
Observation:  Ruff offers a larger rule set and automatic fixing of lint violations, as well as the ability to replace other tools such as Black, isort, yesqa, eradicate, and pyupgrade. It also supports all Python versions from 3.7 onwards and does not require the installation of Rust.
Thought: This information is helpful, but I should also consider the specific needs of my project and team.
Action: State of Union QA System
Action Input: "What are the unique features of ruff compared to flake8?"
Observation:  Ruff has over 800 implemented rules, compared to Flake8's ~409. Ruff also has the ability to automatically fix its own lint violations, while Flake8 does not. Additionally, Ruff does not support custom or third-party rules, while Flake8 does.
Thought: This is us

{'input': 'Why use ruff over flake8?',
 'output': 'Ruff may be a better choice for projects that require a larger rule set and automatic fixing of lint violations, but it may also have compatibility issues and a steeper learning curve compared to flake8. Ultimately, the decision should be based on the specific needs and preferences of the project and team.'}

## Use the Agent solely as a router

You can also set `return_direct=True` if you intend to use the agent as a router and just want to directly return the result of the RetrievalQAChain.

Notice that in the above examples the agent did some extra work after querying the RetrievalQAChain. You can avoid that and just return the result directly.

In [45]:
tools = [
    Tool(
        name="Ruff Documentation QA System",
        func=ruff.run,
        description="Useful for answering questions about the Ruff Python linter documentation.",
        return_direct=True,
    ),
    Tool(
        name="Python Linter Information",
        func=ruff.run,
        description="Use this when the question is about Python linting, Ruff features, configuration, or usage.",
        return_direct=True,
    ),
]

In [46]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [47]:
agent.invoke("What does Ruff do for Python developers?")



> Entering new AgentExecutor chain...
 Ruff is a Python linter, so it likely helps with code analysis and identifying potential errors or style issues.
Action: Python Linter Information
Action Input: None
Observation:  I don't know.


> Finished chain.


{'input': 'What does Ruff do for Python developers?',
 'output': " I don't know."}

In [48]:
agent.invoke("Why use ruff over flake8?")



> Entering new AgentExecutor chain...
 I should compare the features and benefits of both tools.
Action: Python Linter Information
Action Input: ruff, flake8
Observation:  Ruff can replace Flake8 when used with any of the following plugins: Black, isort, yesqa, eradicate, and most of the rules implemented in pyupgrade.


> Finished chain.


{'input': 'Why use ruff over flake8?',
 'output': ' Ruff can replace Flake8 when used with any of the following plugins: Black, isort, yesqa, eradicate, and most of the rules implemented in pyupgrade.'}

## Multi-Hop vector store reasoning

Because vector stores are easily usable as tools in agents, it is easy to use answer multi-hop questions that depend on vector stores using the existing agent framework.

In [49]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

In [50]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [51]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?"
)



> Entering new AgentExecutor chain...
 I should check if the president mentioned the tool in the state of the union before answering the first question.
Action: State of Union QA System
Action Input: What tool did the president mention in the state of the union?
Observation:  I don't know.
Thought: I should now check if ruff uses that tool to run over Jupyter Notebooks.
Action: Ruff QA System
Action Input: What tool does ruff use to run over Jupyter Notebooks?
Observation:  Ruff uses nbQA, a tool for running linters and code formatters over Jupyter Notebooks.
Thought: I now know the final answer.
Final Answer: Ruff uses nbQA to run over Jupyter Notebooks. The president did not mention this tool in the state of the union.

> Finished chain.


{'input': 'What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?',
 'output': 'Ruff uses nbQA to run over Jupyter Notebooks. The president did not mention this tool in the state of the union.'}